In [ ]:
import re
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Load the Naukri Jobs Dataset

In [ ]:
df = pd.read_csv("/content/marketing_sample_for_naukri_com-jobs__20190701_20190830__30k_data.csv")
print("Shape:", df.shape)
df.head(3)

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(subset=["Job Title", "Key Skills", "Functional Area"], inplace=True)

In [ ]:
df.duplicated().sum()

In [ ]:
df.dropna(subset=["Key Skills", "Job Title", "Functional Area"], inplace=True)
print("Shape after dropping missing values:", df.shape)

## 2. Clean Text

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)          # strip any stray HTML
    text = text.replace("|", " ")                 # Key Skills are pipe-separated
    text = re.sub(r"[^a-z0-9\s]", " ", text)      # keep letters/numbers
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
# Combine job title + key skills into a single text field (our "review" equivalent)
df["text"] = df["Job Title"].astype(str) + " " + df["Key Skills"].astype(str)
df["clean_text"] = df["text"].apply(clean_text)

## 3. Build the Target: Functional Area (Top 15 + Other)

In [ ]:
# The raw "Functional Area" column has 70+ categories with a long tail.
# Keep the top 15 and bucket everything else into "Other" (matches the site's own convention).
TOP_N = 15
top_classes = df["Functional Area"].value_counts().head(TOP_N).index.tolist()
df["target"] = df["Functional Area"].where(df["Functional Area"].isin(top_classes), "Other")

le = LabelEncoder()
df["label"] = le.fit_transform(df["target"])
print("Classes:", list(le.classes_))

## 4. Train/Test Split

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["clean_text"], df["label"],
    test_size=0.2, random_state=RANDOM_STATE, stratify=df["label"]
)

## 5. TF-IDF Features + Logistic Regression Baseline

In [ ]:
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

In [ ]:
baseline = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
baseline.fit(X_train_tfidf, y_train)

## 6. Evaluate

In [1]:
preds = baseline.predict(X_test_tfidf)
print("Test accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds, target_names=le.classes_))

Test accuracy: 0.7212

                                                              precision    recall  f1-score   support

        Accounts , Finance , Tax , Company Secretary , Audit       0.80      0.83      0.82       265
                                    Engineering Design , R&D       0.67      0.35      0.46        92
      Financial Services , Banking , Investments , Insurance       0.59      0.35      0.44       133
                      HR , Recruitment , Administration , IR       0.86      0.80      0.83       271
         IT Software - Application Programming , Maintenance       0.72      0.84      0.78      1441
                                     IT Software - ERP , CRM       0.71      0.43      0.54        92
                                         IT Software - Other       0.00      0.00      0.00        83
                                  IT Software - QA & Testing       0.63      0.56      0.59        81
      ITES , BPO , KPO , LPO , Customer Service , Operatio

## 7. Save Artifacts

In [ ]:
with open("logreg_model.pkl", "wb") as f:
    pickle.dump(baseline, f)
with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

In [ ]:
from google.colab import files

files.download("logreg_model.pkl")
files.download("tfidf_vectorizer.pkl")
files.download("label_encoder.pkl")